# Progetto di Machine Learning Classico
## Modulo 1: Unsupervised Feature Extraction per la Predizione del PEGI dei Videogiochi

Questo notebook implementa la prima fase del progetto d'esame. L'obiettivo principale è l'estrazione non supervisionata di feature cromatiche dalle immagini di copertina dei videogiochi di Steam (`header_image`), per utilizzarle successivamente come predittori insieme a feature tabulari (prezzo e genere) per predire la classificazione di età PEGI.

### Obiettivi del Modulo 1:
1. **Configurazione ambiente**: Installazione delle dipendenze necessarie, inclusa la libreria ufficiale `kagglehub` per scaricare i dati senza necessità di configurare manualmente credenziali locali.
2. **Download Dataset**: Recupero del dataset `fronkongames/steam-games-dataset` tramite API `kagglehub`.
3. **Data Cleaning**: Rimozione dei record non classificati (`required_age == 0`) e di quelli sprovvisti di immagini valide. Mappatura delle età sui target PEGI ufficiali (3, 7, 12, 16, 18).
4. **K-Means Clustering per Feature Extraction**:
   - Addestramento di un modello **K-Means** globale nello spazio colore **CIE L*a*b*** su un campione rappresentativo di pixel. Salvataggio del modello in un file `.pkl`.
   - Segmentazione delle immagini e calcolo dell'istogramma delle frequenze dei cluster per singola immagine.
   - Generazione di un vettore ordinato di 15 feature cromatiche (le coordinate L, A, B dei 5 cluster ordinati per dominanza decrescente).
5. **Esportazione e Analisi**: Salvataggio del dataset finale (`steam_features_dataset.csv`) e verifica del bilanciamento delle classi PEGI tramite un grafico di distribuzione.

In [1]:
# Installazione dei pacchetti richiesti per l'esecuzione del notebook
!pip install -q kagglehub tqdm opencv-python pillow scikit-learn pandas numpy requests seaborn matplotlib scikit-image

import os
print("[INFO] Librerie installate con successo. Pronto per il download dei dati.")

[INFO] Librerie installate con successo. Pronto per il download dei dati.


## Sezione 2: Download ed Estrazione del Dataset Steam tramite KaggleHub

Utilizziamo la libreria `kagglehub` per scaricare direttamente l'ultima versione del dataset `"fronkongames/steam-games-dataset"`. Questo metodo permette di scaricare dataset pubblici di Kaggle in modo rapido e senza la necessità di configurare credenziali o chiavi API personali.

In [2]:
import kagglehub
import os

print("[INFO] Avvio del download del dataset tramite kagglehub...")
# Download dell'ultima versione del dataset di Steam
dataset_path = kagglehub.dataset_download("fronkongames/steam-games-dataset")

print("Path del dataset scaricato:", dataset_path)

# Elenco dei file presenti nella directory per confermare il successo del download
downloaded_files = os.listdir(dataset_path)
print("[INFO] File presenti nella cartella:")
for file in downloaded_files:
    print(f" - {file}")

[INFO] Avvio del download del dataset tramite kagglehub...
Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Path del dataset scaricato: /kaggle/input/steam-games-dataset
[INFO] File presenti nella cartella:
 - games.csv
 - games.json


## Sezione 3: Caricamento e Pulizia dei Dati (Data Cleaning e Mappatura PEGI)

In questa sezione carichiamo il dataset tabularmente ed applichiamo i filtri necessari per la nostra pipeline:
- Esclusione dei videogiochi senza classificazione d'età (`required_age == 0`).
- Validazione formale dell'URL delle immagini presenti nella colonna `header_image`.
- Selezione e mappatura dei record la cui età corrisponde esattamente ai target PEGI ufficiali standard: **3, 7, 12, 16, 18**.

In [4]:
import pandas as pd
import numpy as np
import os

# Identificazione dinamica del file CSV nella cartella scaricata da kagglehub
csv_files = [file for file in os.listdir(dataset_path) if file.endswith('.csv')]

if len(csv_files) > 0:
    csv_file_path = os.path.join(dataset_path, csv_files[0])
    print(f"[INFO] Caricamento del dataset da: {csv_file_path}")
    raw_df = pd.read_csv(csv_file_path)
    raw_df.columns = raw_df.columns.str.lower().str.replace(' ', '_')
else:
    raise FileNotFoundError("[ERROR] Nessun file CSV rilevato nella directory del dataset scaricato.")

print(f"[INFO] Righe totali caricate: {raw_df.shape[0]}")

# 1. Filtro dei giochi non classificati (required_age <= 0)
df_filtered = raw_df[raw_df['required_age'] > 0].copy()
print(f"[INFO] Record rimanenti dopo il filtro required_age > 0: {df_filtered.shape[0]}")

# 2. Filtraggio e validazione degli URL delle immagini di copertina (header_image)
df_filtered = df_filtered[df_filtered['header_image'].notna()]
df_filtered = df_filtered[df_filtered['header_image'].str.strip() != '']
# Utilizzo di regex per verificare che la colonna contenga un URL HTTP/HTTPS ben formato
df_filtered = df_filtered[df_filtered['header_image'].str.match(r'^https?://', na=False)]
print(f"[INFO] Record rimanenti dopo la validazione degli URL immagine: {df_filtered.shape[0]}")

# 3. Mappatura e filtraggio secondo le classi ufficiali PEGI (3, 7, 12, 16, 18)
official_pegi = {3, 7, 12, 16, 18}
# Manteniamo esclusivamente le righe la cui età coincide con un target PEGI ufficiale
df_filtered = df_filtered[df_filtered['required_age'].isin(official_pegi)].copy()
df_filtered.rename(columns={'required_age': 'pegi'}, inplace=True)

print(f"[INFO] Record finali puliti pronti per l'estrazione: {df_filtered.shape[0]}")
print("\n[INFO] Distribuzione per categoria PEGI:")
print(df_filtered['pegi'].value_counts())

[INFO] Caricamento del dataset da: /kaggle/input/steam-games-dataset/games.csv
[INFO] Righe totali caricate: 122611


KeyError: 'required_age'

## Sezione 4: Pipeline Unsupervised per l'Estrazione di Feature tramite K-Means Globale

Per catturare la composizione cromatica delle copertine dei videogiochi, utilizziamo un approccio non supervisionato:
1. **Campionamento e Addestramento**: Scarichiamo un piccolo campione di immagini, le convertiamo nello spazio colore **CIE L*a*b*** (molto più coerente con la percezione umana del colore rispetto a RGB) ed addestriamo un modello **K-Means** globale con $k=5$.
2. **Elaborazione e Ordinamento Frequenze**: Per ciascuna immagine elaborata, ordiniamo i centroidi globali dal colore predominante (più frequente) al meno frequente in base all'istogramma locale. Questo assicura l'invarianza spaziale della feature di colore dominante.
3. **Flattizzazione**: Ogni immagine viene mappata in un vettore di 15 feature ($5 \text{ centroidi} \times 3 \text{ canali LAB}$): L1, A1, B1 ... L5, A5, B5.
4. **Metadati**: Ad ogni riga vengono associati il prezzo (`price`) e i generi del gioco (`genres`).

In [ ]:
import cv2
import requests
import io
import joblib
import urllib3
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from skimage.feature import local_binary_pattern
from tqdm import tqdm

# Disabilitazione di avvisi SSL relativi a richieste HTTP non verificate
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- FASE 1: Addestramento del K-Means globale ---
print("[INFO] Fase 1: Addestramento del K-Means Globale per i Centroidi Cromatici...")

# Selezione di un campione di 50 giochi per catturare una gamma cromatica diversificata
n_train_samples = 50
training_sample_df = df_filtered.sample(n=min(n_train_samples, len(df_filtered)), random_state=42)
sampled_pixel_list = []

for idx, row in tqdm(training_sample_df.iterrows(), total=len(training_sample_df), desc="Download immagini di training"):
    try:
        response = requests.get(row['header_image'], timeout=10, verify=False)
        if response.status_code == 200:
            # Caricamento e conversione dell'immagine
            image_raw = Image.open(io.BytesIO(response.content)).convert('RGB')
            image_bgr = cv2.cvtColor(np.array(image_raw), cv2.COLOR_RGB2BGR)
            
            # Conversione allo spazio colore LAB e ridimensionamento
            image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
            image_resized = cv2.resize(image_lab, (300, 200)) # Larghezza 300, Altezza 200
            
            # Campionamento casuale di pixel dall'immagine per evitare sovraccarico di memoria
            all_image_pixels = image_resized.reshape(-1, 3)
            selected_indices = np.random.choice(all_image_pixels.shape[0], size=min(1000, all_image_pixels.shape[0]), replace=False)
            sampled_pixel_list.append(all_image_pixels[selected_indices])
    except Exception:
        continue

if len(sampled_pixel_list) == 0:
    raise ValueError("[ERROR] Impossibile scaricare le immagini campione. Controlla la connessione di rete.")

# Aggregazione di tutti i pixel campionati
all_sampled_pixels = np.vstack(sampled_pixel_list)
print(f"[INFO] Dataset di addestramento K-Means pronto. Pixel totali: {all_sampled_pixels.shape[0]}")

# Addestramento dell'algoritmo K-Means per identificare i 5 cluster cromatici principali del dataset
kmeans_global = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_global.fit(all_sampled_pixels)

# Salvataggio del modello addestrato su file locale per inferenza futura
model_save_path = 'kmeans_model.pkl'
joblib.dump(kmeans_global, model_save_path)
print(f"[INFO] Modello K-Means globale salvato con successo in '{model_save_path}'.")

# --- FASE 2: Estrazione Feature d'Élite (Spatial Grid, Luminosità, Texture, Contesti) ---
print("\n[INFO] Fase 2: Estrazione e costruzione del dataset finale delle feature...")

# Caricamento del modello salvato per verificarne l'integrità
loaded_kmeans = joblib.load(model_save_path)

# Vocabolario specifico per l'NLP classico basato su termini critici e positivi correlati al PEGI
pegi_vocabulary = [
    'violence', 'violent', 'gore', 'blood', 'bloody', 'nudity', 'sexual', 'sex', 'sexy',
    'mature', 'adult', 'nsfw', 'horror', 'scary', 'fear', 'dark', 'zombie', 'zombies',
    'weapon', 'weapons', 'gun', 'guns', 'shoot', 'shooter', 'shooting', 'kill', 'killing',
    'combat', 'fight', 'fighting', 'death', 'dead', 'brutal', 'slasher', 'drugs', 'alcohol',
    'profanity', 'language', 'gambling', 'coarse', 'crude', 'explicit', 'erotic', 'hentai',
    'cute', 'family', 'relaxing', 'cartoon', 'cartoons', 'kids', 'kid', 'friendly',
    'wholesome', 'cozy', 'sweet', 'puzzle', 'puzzles', 'colorful', 'funny', 'happy',
    'education', 'educational', 'learning', 'child', 'children', 'peaceful', 'casual',
    'simulation', 'strategy', 'adventure', 'fantasy', 'magic', 'multiplayer', 'exploration'
]

# Selezione del blocco target di 4000 giochi per il download delle immagini in tempo reale
target_batch_size = 4000
batch_df = df_filtered.sample(n=min(target_batch_size, len(df_filtered)), random_state=42).copy()

extracted_dataset_rows = []

# Dimensioni griglia spaziale (3x3)
grid_rows = 3
grid_cols = 3
img_h, img_w = 200, 300
h_step = img_h // grid_rows
w_step = img_w // grid_cols

for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc="Processing immagini e metadati"):
    try:
        response = requests.get(row['header_image'], timeout=5, verify=False)
        if response.status_code == 200:
            # Lettura dell'immagine e conversione in formato LAB e Grayscale
            image_raw = Image.open(io.BytesIO(response.content)).convert('RGB')
            image_bgr = cv2.cvtColor(np.array(image_raw), cv2.COLOR_RGB2BGR)
            image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
            image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
            
            # Ridimensionamento uniforme dell'immagine
            image_resized = cv2.resize(image_lab, (img_w, img_h))
            
            # Costruzione del record base
            record = {
                'price': row.get('price', 0.0),
                'genres': row.get('genres', ''),
                'pegi': row.get('pegi'),
                '_text_content': f"{str(row.get('about_the_game', ''))} {str(row.get('tags', ''))}"
            }
            
            # 1. Feature di Luminosità e Contrasto Globale (canale L)
            L_channel_global = image_resized[:, :, 0]
            record['L_mean_global'] = float(np.mean(L_channel_global))
            record['L_std_global'] = float(np.std(L_channel_global))
            
            # 2. Spatial Grid Clustering (3x3 zone) & Luminosità Locale
            zone_idx = 0
            for r in range(grid_rows):
                y_start = r * h_step
                y_end = img_h if r == grid_rows - 1 else (r + 1) * h_step
                for c in range(grid_cols):
                    x_start = c * w_step
                    x_end = img_w if c == grid_cols - 1 else (c + 1) * w_step
                    
                    # Sotto-immagine per la zona corrente
                    zone_pixels_lab = image_resized[y_start:y_end, x_start:x_end]
                    zone_pixels_flat = zone_pixels_lab.reshape(-1, 3)
                    
                    # Quantizzazione cromatica via K-Means globale
                    zone_assignments = loaded_kmeans.predict(zone_pixels_flat)
                    frequencies = np.bincount(zone_assignments, minlength=5)
                    frequencies_rel = frequencies / len(zone_assignments)
                    
                    # Salvataggio delle 5 frequenze cromatiche per la zona
                    for cluster_idx in range(5):
                        record[f'grid_z{zone_idx}_c{cluster_idx}'] = float(frequencies_rel[cluster_idx])
                    
                    # Feature di Luminosità Locale (canale L della zona)
                    L_channel_zone = zone_pixels_lab[:, :, 0]
                    record[f'L_mean_z{zone_idx}'] = float(np.mean(L_channel_zone))
                    record[f'L_std_z{zone_idx}'] = float(np.std(L_channel_zone))
                    
                    zone_idx += 1
                    
            # 3. Descrittore di Texture (Local Binary Patterns - LBP)
            # Parametri LBP uniformi per invarianza rotazionale
            radius = 1
            n_points = 8 * radius
            lbp = local_binary_pattern(image_gray, n_points, radius, method='uniform')
            # n_bins per il metodo 'uniform' con P=8 è 10
            lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
            for bin_idx in range(10):
                record[f'lbp_hist_b{bin_idx}'] = float(lbp_hist[bin_idx])
                
            extracted_dataset_rows.append(record)
    except Exception as e:
        continue

print(f"\n[INFO] Elaborazione immagini completata. Record validi estratti: {len(extracted_dataset_rows)}")

# --- FASE 3: Text Mining TF-IDF ---
if len(extracted_dataset_rows) > 0:
    print("[INFO] Fase 3: Applicazione di TF-IDF sulle descrizioni e tag dei giochi...")
    # Creazione DataFrame temporaneo
    temp_df = pd.DataFrame(extracted_dataset_rows)
    
    # Inizializzazione TF-IDF con il nostro vocabolario predefinito
    tfidf = TfidfVectorizer(vocabulary=pegi_vocabulary, lowercase=True, stop_words='english')
    tfidf_matrix = tfidf.fit_transform(temp_df['_text_content'].fillna(''))
    
    # Creazione del DataFrame per le feature testuali
    tfidf_columns = [f'tfidf_{word}' for word in pegi_vocabulary]
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_columns, index=temp_df.index)
    
    # Rimozione colonna di testo temporanea e concatenazione di TF-IDF
    final_feature_df = pd.concat([temp_df.drop(columns=['_text_content']), tfidf_df], axis=1)
    
    # Salvataggio del Vectorizer TF-IDF su disco
    tfidf_save_path = 'tfidf_vectorizer.pkl'
    joblib.dump(tfidf, tfidf_save_path)
    print(f"[INFO] Vectorizer TF-IDF salvato correttamente in '{tfidf_save_path}'.")
else:
    raise ValueError("[ERROR] Nessun record estratto con successo.")

: 

## Sezione 5: Consolidamento, Esportazione del Dataset di Feature e Analisi del Bilanciamento delle Classi

In quest'ultima sezione:
1. Creiamo un DataFrame Pandas contenente le feature estratte per ogni gioco.
2. Salviamo il file in formato CSV pronto per l'utilizzo nel modulo successivo di modellazione predittiva.
3. Rappresentiamo la distribuzione della variabile target PEGI tramite un grafico a barre di Seaborn per valutare visivamente il bilanciamento delle classi ed eventuali necessità di tecniche di campionamento.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Il DataFrame final_feature_df è già stato creato ed arricchito con TF-IDF nella cella precedente
print(f"[INFO] Dimensioni del dataset finale delle feature: {final_feature_df.shape}")
print("\n[INFO] Prime 5 righe del dataset estratto:")
print(final_feature_df.head())

# Salvataggio su disco nella directory designata per Google Colab
output_csv_path = 'steam_features_dataset.csv'
final_feature_df.to_csv(output_csv_path, index=False)
print(f"\n[INFO] Dataset salvato con successo in: {output_csv_path}")

# Plot della distribuzione della classificazione PEGI
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Conteggio grafico dei PEGI
ax = sns.countplot(
    data=final_feature_df, 
    x='pegi', 
    hue='pegi', 
    palette='viridis', 
    legend=False
)

# Configurazione testi e titoli del grafico
plt.title('Distribuzione della Variabile Target PEGI nel Dataset Elaborato', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Classificazione PEGI (Età Richiesta)', fontsize=12)
plt.ylabel('Frequenza Assoluta (Conteggio)', fontsize=12)

# Inserimento dei valori esatti sopra ciascuna barra del grafico
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.annotate(
            f'{int(height)}', 
            (patch.get_x() + patch.get_width() / 2., height),
            ha='center', 
            va='baseline', 
            fontsize=11, 
            color='black', 
            xytext=(0, 5), 
            textcoords='offset points'
        )

plt.tight_layout()
plt.show()